In [ ]:
import os
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../../.env")

ARIREGISTER_URL = "https://ariregxmlv6.rik.ee/"
ARIREGISTER_USER = os.getenv("ARIREGISTER_USER")
ARIREGISTER_PASSWORD = os.getenv("ARIREGISTER_PASSWORD")
NS = "http://arireg.x-road.eu/producer/"

def _soap_request(body_xml):
    """Send a SOAP request and return the parsed XML root."""
    envelope = f"""<?xml version="1.0" encoding="UTF-8"?>
<soapenv:Envelope xmlns:soapenv="http://schemas.xmlsoap.org/soap/envelope/"
  xmlns:prod="{NS}">
  <soapenv:Body>
    {body_xml}
  </soapenv:Body>
</soapenv:Envelope>"""
    resp = requests.post(
        ARIREGISTER_URL,
        data=envelope.encode("utf-8"),
        headers={"Content-Type": "text/xml; charset=utf-8", "SOAPAction": ""},
        timeout=30,
    )
    resp.raise_for_status()
    return ET.fromstring(resp.content)

def list_reports(registry_code):
    """List available annual report types for a company."""
    body = f"""<prod:majandusaastaAruanneteLoetelu_v1>
      <prod:keha>
        <prod:ariregister_kasutajanimi>{ARIREGISTER_USER}</prod:ariregister_kasutajanimi>
        <prod:ariregister_parool>{ARIREGISTER_PASSWORD}</prod:ariregister_parool>
        <prod:ariregistri_kood>{registry_code}</prod:ariregistri_kood>
      </prod:keha>
    </prod:majandusaastaAruanneteLoetelu_v1>"""
    root = _soap_request(body)
    reports = []
    for item in root.iter(f"{{{NS}}}majandusaasta_aruanded"):
        reports.append({
            "aruande_kood": item.findtext(f"{{{NS}}}aruande_kood"),
            "aruande_nimetus": item.findtext(f"{{{NS}}}aruande_nimetus"),
            "aruande_aasta": item.findtext(f"{{{NS}}}aruande_aasta"),
            "majandusaasta_algus": item.findtext(f"{{{NS}}}majandusaasta_algus"),
            "majandusaasta_lopp": item.findtext(f"{{{NS}}}majandusaasta_lopp"),
        })
    return pd.DataFrame(reports)

def fetch_report(registry_code, report_type, year, lang="est"):
    """Fetch a single annual report and return as a DataFrame."""
    body = f"""<prod:majandusaastaAruanneteKirjed_v1>
      <prod:keha>
        <prod:ariregister_kasutajanimi>{ARIREGISTER_USER}</prod:ariregister_kasutajanimi>
        <prod:ariregister_parool>{ARIREGISTER_PASSWORD}</prod:ariregister_parool>
        <prod:ariregistri_kood>{registry_code}</prod:ariregistri_kood>
        <prod:aruande_liik>{report_type}</prod:aruande_liik>
        <prod:aruandeaasta>{year}</prod:aruandeaasta>
        <prod:keel>{lang}</prod:keel>
      </prod:keha>
    </prod:majandusaastaAruanneteKirjed_v1>"""
    root = _soap_request(body)
    rows = []
    for row in root.iter(f"{{{NS}}}majandusaasta_aruanded_read"):
        row_data = {
            "rea_nr": row.findtext(f"{{{NS}}}rea_nr"),
            "rea_nimetus": row.findtext(f"{{{NS}}}rea_nimetus"),
        }
        for col in row.iter(f"{{{NS}}}majandusaasta_aruanded_veerud"):
            col_name = col.findtext(f"{{{NS}}}veeru_nimetus")
            col_value = col.findtext(f"{{{NS}}}vaartus")
            if col_name and col_value:
                row_data[col_name] = int(col_value)
        rows.append(row_data)
    return pd.DataFrame(rows)

print("Ready — helpers loaded")